# Lab 7 — Grad-CAM: Explainable AI

**Student:** Hitesh Gupta | **Roll No.:** IC-2k23-39

Gradient-weighted Class Activation Mapping (Grad-CAM) visualises which
regions of an input image contribute most to a CNN's prediction.
Uses a pre-trained ResNet-18 from torchvision.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
os.makedirs('outputs', exist_ok=True)

## 1. Load Pre-trained ResNet-18

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights).to(device)
model.eval()

# ImageNet preprocessing
preprocess = weights.transforms()
categories = weights.meta["categories"]
print('Num classes:', len(categories))
print('Sample classes:', categories[:5])

## 2. Grad-CAM Implementation

In [ ]:
class GradCAM:
    """Grad-CAM for a target layer of a CNN."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, input_tensor, target_class=None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        score = output[0, target_class]
        score.backward()

        # Global-average-pool gradients -> neuron importance weights
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:],
                            mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_class, output

# Target the last convolutional layer of ResNet-18
target_layer = model.layer4[-1]
gradcam = GradCAM(model, target_layer)
print('Grad-CAM initialised on layer4[-1]')

## 3. Run on Sample Images

In [ ]:
def load_image(path):
    from PIL import Image
    img = Image.open(path).convert('RGB')
    return preprocess(img).unsqueeze(0).to(device), img

def overlay_cam(img_pil, cam, alpha=0.5):
    cam_uint8 = (cam * 255).astype('uint8')
    cam_color = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)
    cam_color = cv2.cvtColor(cam_color, cv2.COLOR_BGR2RGB)
    img_np = np.array(img_pil.resize((cam.shape[1], cam.shape[0])))
    return (alpha * cam_color + (1 - alpha) * img_np).astype('uint8')

# Try to find a sample image; otherwise create a synthetic one
sample_paths = [
    'inputs/sample1.jpg', 'inputs/sample2.jpg',
    '../datasets/sample1.jpg',
]
found = [p for p in sample_paths if os.path.exists(p)]

if found:
    paths = found[:3]
else:
    # Synthetic test image (random noise normalised)
    print('No sample images found — using synthetic input.')
    paths = []

results = []
for path in paths:
    inp, img_pil = load_image(path)
    cam, cls_idx, logits = gradcam(inp)
    cls_name = categories[cls_idx]
    prob = F.softmax(logits, dim=1)[0, cls_idx].item()
    overlay = overlay_cam(img_pil, cam)
    results.append((img_pil, overlay, cls_name, prob))
    print(f'{os.path.basename(path)} -> {cls_name} ({prob:.3f})')

In [ ]:
if results:
    fig, axes = plt.subplots(len(results), 3, figsize=(12, 4 * len(results)))
    if len(results) == 1:
        axes = axes.reshape(1, -1)
    for i, (orig, over, cls, prob) in enumerate(results):
        axes[i, 0].imshow(orig)
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(over)
        axes[i, 1].set_title(f'Grad-CAM: {cls} ({prob:.2f})')
        axes[i, 1].axis('off')
        axes[i, 2].imshow(cam, cmap='jet')
        axes[i, 2].set_title('CAM Heatmap')
        axes[i, 2].axis('off')
    plt.tight_layout()
    plt.savefig('outputs/gradcam.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Add sample images to inputs/ to visualise Grad-CAM.')

## Theory

Grad-CAM computes a coarse localisation map highlighting important
regions:

1. Forward pass → activations `A^k` of target conv layer.
2. Backward pass → gradients `∂y^c / ∂A^k`.
3. Global-average-pool gradients → neuron importance weights `α_k`.
4. Weighted combination + ReLU → class-discriminative heatmap.

`L^c = ReLU( Σ_k α_k^c A^k )`